# Image Generation

> 🎯 **本章學完你將能學會什麼：**
> - 理解 OpenAI 的 **GPT-Image-1** 模型特色與使用方式  
> - 熟悉 `client.images.generate()` 與 `client.images.edit()` 的使用方法  
> - 掌握如何設定影像生成的 **size**、**quality**、**moderation** 等參數  
> - 能以英文 prompt 生成高品質 AI 圖像  
> - 瞭解如何將 base64 圖像資料轉換、顯示與儲存  

OpenAI 提供 文生圖 (text to image) 和 圖生圖 (image to image) API。

## GPT-Image-2

OpenAI 最新的文生圖模型 (2026-04-21 發布)

優點:
- 品質更好 (quality `medium` 的結果 比 Image-1.5 quality `high` 更好)
- 價格更便宜 
- 可以穩定出文字(英文)

建議使用英文作為Prompt。

## 📚 延伸資源與參考連結

> 這些連結可幫助你更深入理解 OpenAI API 與 LangChain 的應用方式，  
> 特別適合在閱讀教材後進一步查閱官方技術文件與實作範例。

---

### 🧠 OpenAI 相關文件

- [**OpenAI API Image Generation 文檔**](https://platform.openai.com/docs/guides/image-generation)  
  說明如何使用 `gpt-image-1` 模型進行 **文字轉圖 (Text-to-Image)**、**圖像編輯 (Image-to-Image)** 與 **Inpainting**。  
  包含請求參數（如 `size`、`quality`、`response_format`）及回傳格式的詳細說明。  

- [**OpenAI API 參考文檔 (API Reference)**](https://platform.openai.com/docs/api-reference/images)  
  列出所有影像生成／編輯端點與可用參數，並提供實際範例程式碼。  
  適合在開發階段查詢 API 請求格式與欄位說明。  

- [**OpenAI Cookbook**](https://cookbook.openai.com/)  
  官方技術範例集（recipes），展示如何整合 OpenAI API 至各種應用場景。  
  其中的 [影像生成範例](https://cookbook.openai.com/examples/generate_images_with_gpt_image)  
  對應到本章節的 `client.images.generate()` 教學。


- model: gpt-image-1
    - size (str): 1024x1024 (square), 1536x1024 (landscape), 1024x1536 (portrait) or auto (default)
    - quality: low, medium, high or auto
    - moderation: auto, low

In [ ]:
import os

os.chdir("../../")

範例

In [ ]:
from openai import OpenAI

from initialization import credential_init

credential_init()

client = OpenAI()

In [ ]:
import base64

from IPython.display import display, HTML

prompt = ("A Sumi-e style watercolor painting of mountains during sunset. The sky is depicted with bold "
          "splashes of orange, pink, and purple hues, blending and overlapping in a dynamic composition. "
          "The mountains are represented with expressive brushstrokes, emphasizing their majestic and serene "
          "presence. The focus is on capturing the essence and mood of the scene rather than detailed realism. "
          "The overall effect is serene and contemplative, with a harmonious balance of color and form.")

response = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1024",
    quality='medium',
    n=1,
    # response_format = 'b64_json'
)

image_base64 = response.data[0].b64_json

# 將返回的 base64字串轉換為圖像並且儲存
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

可以出中文在我預料之外...

## 挑戰：如何有效地撰寫 Text-to-Image 提示詞：

> 🎯 **本章學完你將能學會什麼：**
> - 分辨兩種提示詞風格：**標籤式提示 (Danbooru Tags)** 與 **自然語言提示 (Natural Language Prompts)**  
> - 理解不同模型對標籤的支援度差異  
> - 學習如何結合藝術、攝影用語提升生成品質   


在使用 AI 生成圖像（例如 OpenAI 的 Image-2）時，提示詞（prompt）的寫法對結果有決定性影響。主要有兩種提示詞格式：

- 標籤式提示（Danbooru Tag):

    - 範例:    

        masterpiece, best quality, beautiful eyes, clear eyes, detailed eyes, Blue-eyes, 1girl, 20_old, full-body, break, smoking, break, high_color, blue-hair, beauty, black-boots,break, break, Flat vector art, Colorful art, white_shirt, simple_background, blue_background, Ink art, peeking out upper body, Eyes


    - 特點與注意事項：

        - 生效與否取決於模型，不同模型對同一個標籤的理解可能不同。
        - 某些標籤是通用的，例如 1girl、ulzzang，但呈現效果可能差異很大。
        - 一些標籤需要專業知識，例如 chiaroscuro（明暗對照法）。
        - 需要多次嘗試與微調，才能找到最佳組合。

2. 自然語言提示（Natural Language Prompt):

    - 範例:

       A Japanese idol with a breathtakingly glamorous ulzzang appearance,  She has a slim, v-shaped face with large, almond-shaped eyes that sparkle with a lustrous, captivating charm, exuding an aura of youth and ethereal beauty. Her expression is innocent yet alluring, with flawless porcelain skin that enhances her delicate, anime-inspired features. The setting is carefully crafted to complement her enchantment, with soft, diffused lighting that accentuates her mesmerizing, glamorous presence, creating a dreamy and youthful, anime-like allure.


    - 特點與注意事項：

        - 句子寫得流暢、語言優美，能提升生成圖像的質感。

        - 對非母語使用者來說，整合多個描述性詞彙是一大挑戰。

        - 部分詞彙在監控嚴格的模型下可能會被屏蔽，例如 serafuku。

        - Image-1 等模型可能會對過於明顯的 NSFW 提示詞進行攔截。若想生成 NSFW 的內容，建議可以參考開源社群，例如 TensorArt/TensorHub。

## 融入 LCEL 與 LangChain — 讓模型幫你生成更好的 Prompt

> 🎯 **本章學完你將能學會什麼：**
> - 能將 GPT 模型生成的文字提示直接導入影像生成 API  
> - 實作一個自動從描述文字 → prompt → 圖像的完整流程  

本節將介紹如何運用 LangChain 建立一個能自動化撰寫提示詞的流程，並串接影像生成 API，達成端到端的自動圖像生成。

### Step1

可以給予內容，並且讓文字模型幫忙寫提示詞。並且可以考慮使用mlflow監視產出的提示詞

In [ ]:
from textwrap import dedent

from langchain_ollama import ChatOllama
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser


def build_standard_chat_prompt_template(kwargs):
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)
    
    return chat_prompt_template


model = ChatOllama(model='deepseek-v3.1:671b-cloud',
                   base_url='https://ollama.com',
                   name='Prompt Generator', temperature=0)

system_template = dedent("""
**Goal:**  
Generate a detailed and descriptive GPT-Image-1 prompt that accurately captures the essence of a provided image description, resulting in a breathtaking masterpiece.

**Input:**  
<IMAGE DESC>:A description of an image (provided by the user).

**Task:**  
Rewrite the input description into a highly detailed, evocative, and technically precise prompt optimized for the GPT-Image-1 model, ensuring it aligns with artistic principles of photography and illustration.

**Rules:**  
- The prompt must be rich in visual detail (e.g., style, composition, lighting, mood, subject, color palette).  
- Avoid vague or abstract language; use concrete and specific terms.  
- Incorporate expertise in photography and illustration techniques (e.g., lens type, perspective, artistic movement).  
- Ensure the prompt is structured to guide the model effectively toward the desired output.  

**Chain of Thought:**  
1. Analyze the input description to identify key elements (subject, setting, style, emotion).  
2. Expand each element with precise artistic and technical details (e.g., "vibrant sunset" → "dramatic golden-hour lighting with long shadows and warm hues").  
3. Structure the prompt logically (e.g., start with subject and action, then environment, style, and technical specs).  
4. Refine the language to be evocative and model-friendly (avoid ambiguity, use descriptive adjectives).  
5. Validate against rules to ensure completeness and effectiveness.
""")

human_template = "<IMAGE DESC>: {image_desc}"

input_ = {"system": {"template": system_template},
          "human": {"template": human_template,
                    "input_variable": ["image_desc"]}}
    
chat_prompt_template = build_standard_chat_prompt_template(input_)

nl_prompt_generation_chain = chat_prompt_template | model | StrOutputParser()

### Step2

將生成的提示詞放入影像生成API中

In [ ]:
from operator import itemgetter
from typing import Dict

from langchain_core.runnables import chain, RunnableLambda, RunnableParallel, RunnablePassthrough


@chain
def gpt_image_worker(kwargs: Dict):

    """
    Generates an image using OpenAI's GPT-Image-1 model based on the provided prompt and optional parameters.
    
    Parameters:
    kwargs (Dict): A dictionary containing the following keys:
        - 'nl_prompt' (str): The natural language prompt describing the image to be generated.
        - 'size' (str, optional): The size of the generated image. Default is "1024x1024".
        - 'quality' (str, optional): The quality of the generated image. Default is "medium".
    
    Returns:
    str: image base64 string
    """
    
    print("Start generating image...")
    print(f"prompt: {kwargs['nl_prompt']}")
    client = OpenAI()

    response = client.images.generate(
        model=kwargs.get('model', "gpt-image-2"),
        prompt=kwargs['nl_prompt'],
        size=kwargs.get("size", "1024x1024"),
        quality=kwargs.get('quality', 'medium'),
        moderation=kwargs.get('moderation', 'auto'),
        n=1)

    image_base64 = response.data[0].b64_json
    
    return image_base64


@chain
def base64_to_file(kwargs):

    """
    Save the image from a base64 string
    """
    
    image_base64 = kwargs['image_base64']
    filename = kwargs['filename']
    
    with open(f"{filename}", "wb") as fh:
        fh.write(base64.b64decode(image_base64))

    return image_base64

In [ ]:
# step 1: 生成依照你想要的圖像描述圖像提示詞
step_1 = RunnablePassthrough.assign(nl_prompt=itemgetter('image_desc')|nl_prompt_generation_chain)

# step 2: 生成圖像，並將base64字串放入image_base64變數中
step_2 = RunnablePassthrough.assign(image_base64=gpt_image_worker)

# step 3: 將base64字串儲存為圖像
step_3 = base64_to_file

# 將三個步驟由水管符號(|)連結起來
gpt_image_chain =  step_1|step_2|step_3

In [ ]:
from textwrap import dedent

image_base64_example = gpt_image_chain.invoke({"size": "1024x1536",
                     "quality": "medium",
                     "image_desc": dedent("""warhammer 40k, astartes, power armor, chain sword, purity seal, 
                     oil painting, cinematic view, battle field, black templars, sacred light upon the arstartes"""),
                     "filename": "tutorial/week_7/astartes.png"
                    })

In [ ]:
# 將返回的 base64字串轉換為圖像並且儲存
HTML(f'<img src="data:image/png;base64,{image_base64_example}"/>')

# 圖像渲染(Image Render)

「圖像渲染」(Image to Image, 簡稱 Img2Img) 指的是：
在已有圖片的基礎上，搭配新的提示詞 (prompt)，生成另一張風格或內容有所變化的圖片。

## ✨ 特點

1. 輸入與輸出

    - 輸入：一張已有的圖片 + 提示詞

    - 輸出：根據提示詞改造過的圖片

2. 靈活性

    - 可以保留原圖的結構（例如人物姿勢），只改變細節（如髮色、衣服、場景）。

    - 也可以做風格轉換，讓照片變成油畫風、漫畫風、插畫風。

3. 常見應用

    - 修圖：去除背景、修改臉部細節、換衣服。

    - 風格化：將現實照片轉成動漫風、插畫風。

    - 迭代設計：快速嘗試不同的角色服裝、髮型或環境。

    - 局部修改 (Inpainting)：在圖片上指定區域，僅對該區域進行替換或修補。

In [ ]:
# import os

# os.chdir("../../")

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

# Build HTML string
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <img src="Eve_Stellar_Blade.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
from textwrap import dedent

from openai import OpenAI

from initialization import credential_init

credential_init()

client = OpenAI()

prompt = dedent("""
Please rending this image as a realistic photo of a girl cosplaying. A Korean girl with a
slim, v-shaped face with large, almond-shaped eyes that sparkle with captivating charm, exuding 
an aura of youth and ethereal beauty. With flawless skin that enhances her delicate, 
anime-inspired features. The setting is carefully crafted to complement her enchantment, with 
soft, diffused lighting that accentuates her mesmerizing, glamorous presence, creating a dreamy 
and youthful, anime-like allure. Her makeup should resemble the features of K-beauty, such as pale skin tones 
and dewed skin texture. 
""")


image_path = os.path.join("tutorial", "week_7", "Eve_Stellar_Blade.png")

result_edit = client.images.edit(
    model="gpt-image-2",
    image=open(image_path, "rb"), 
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}" />')

In [ ]:
result_edit = client.images.edit(
    model="gpt-image-2",
    image=open(image_path, "rb"), 
    prompt=prompt,
    size="1024x1536",
    quality="high",
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}" />')

你可以使用一張或是多張圖片做為參照物

- Noshiro 能代 (Azur Lane)

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="Noshiro - Spring.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
            <img src="Noshiro - Summer.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
        <div>
            <img src="Noshiro - Fall.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
            <img src="Noshiro - Winter.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
image_1 = os.path.join("tutorial", "week_7", "Noshiro - Spring.png")
image_2 = os.path.join("tutorial", "week_7", "Noshiro - Summer.png")
image_3 = os.path.join("tutorial", "week_7", "Noshiro - Fall.png")
image_4 = os.path.join("tutorial", "week_7", "Noshiro - Winter.png")

result_edit = client.images.edit(
    model="gpt-image-2",
    image=[
        open(image_1, "rb"),
        # open(image_2, "rb"),
        # open(image_3, "rb"),
        # open(image_4, "rb"),
    ],
    prompt=dedent("""
    A high-fashion product photograph advertisement for a luxury perfume bottle, capturing mesmerizing glamour and opulence. 
    The perfume bottle is sculpted from flawless crystal glass with elegant gold accents, resting on a surface of black velvet with subtle folds. 
    The bottle is enveloped in a soft, ethereal mist that diffuses light, creating a dreamy atmosphere. 
    Dramatic chiaroscuro lighting highlights the bottle's contours, with a single spotlight casting soft shadows and producing delicate caustics and specular highlights on the glass. 
    The background is a deep, matte black with a faint gradient of gold, adding depth and sophistication. 
    Shot with a macro lens at a shallow depth of field to emphasize the bottle's details and the texture of the velvet, evoking a sense of intimacy and luxury. 
    The color palette is rich and warm, dominated by gold, amber, and deep black tones, enhancing the sensory appeal of elegance and allure. 
    Style: hyper-realistic, cinematic, with a touch of art deco inspiration.
    """),
    size="1024x1536",
    quality="medium"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

In [ ]:
result_edit = client.images.edit(
    model="gpt-image-2",
    image=[
        open(image_1, "rb"),
    ],
    prompt=dedent("""
    A high-fashion product photograph advertisement for a luxury perfume bottle based on the reference image, capturing mesmerizing glamour and opulence. 
    """),
    size="1024x1536",
    quality="medium"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

### 將不同圖片的內容融合在一起

圖片來源: https://tensor.art/u/629260971684229814

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="maehara-1.jpg" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
            <img src="maehara-2.jpg" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
image_a = os.path.join("tutorial", "week_7", "maehara-1.jpg")
image_b = os.path.join("tutorial", "week_7", "maehara-2.jpg")

In [ ]:
result_edit = client.images.edit(
    model="gpt-image-2",
    image=[
        open(image_a, "rb"),
        open(image_b, "rb"),
    ],
    prompt=dedent("""
    Fusion the two images to create a high definition 8k movie poster with the text as the background. 
    """),
    size="1024x1536",
    quality="high"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

# 教學用投影片內容（正經版）

在觀摩多種不是很正經的範例後，我們來探討一個嚴謹的教學應用案例。

## 圖像生成邏輯流程

本系統根據內容性質自動選擇最適合的視覺化方式：

### 1. LLM 判斷機制
- 輸入一段敘述內容後，由大型語言模型(LLM)進行分析判斷
- 判斷產出的圖片應屬於哪種類型：
  - **概念性插畫** (`GeneralImage`)
  - **精確數學表達** (`MatplotlibImage`)

### 2. 分流處理邏輯
```python
if 判斷為 GeneralImage:
    直接將 image_prompt 送入 GPT-Image-2 API 生成插畫
else:
    根據 image_prompt 產生 matplotlib 作圖代碼
    將代碼送入 GPT-Image-2 API 執行並生成圖表

結構化輸出

In [ ]:
from typing import Literal, Union

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class GeneralImage(BaseModel):    
    name: Literal["general"] = "general"    
    image_prompt: str = Field(description="Use for conceptual, artistic, or illustrative visualizations (analogies, real-world examples, conceptual flows) that aid intuition and cannot be represented by a plot.")
    
class MatplotlibImage(BaseModel):    
    name: Literal["matplotlib"] = "matplotlib"    
    image_prompt: str = Field(description="Use for precise technical visualizations where the exact relationship between variables is the primary teaching point (function plots, data charts, geometric proofs).")
    
class Output(BaseModel):    
    reasoning: str = Field(description="Explanation of why an image is unnecessary (e.g., textual sufficiency, pure symbolic logic, or avoiding cognitive overload).")    
    image_selection: Union[GeneralImage, MatplotlibImage, None] = Field(description="The chosen image type if necessary, otherwise None.")


output_parser = PydanticOutputParser(pydantic_object=Output)
format_instructions = output_parser.get_format_instructions()

系統提示詞

In [ ]:
SYSTEM_PROMPT = dedent("""
# ROLE
You are an expert Instructional Designer and Visual Communications Specialist. 

Your goal is to determine if a slide requires a visualization to enhance learning and, if so, which type of image is most appropriate.

# DECISION LOGIC
1. **GeneralImage**: Choose this if the goal is to provide **intuition** via a metaphor, illustration, or conceptual diagram.
2. **MatplotlibImage**: Choose this if the goal is to provide **precision** via a mathematical plot, data graph, or technical chart.
3. **Output (No Image)**: Choose this if the goal is best achieved through **clarity** via concise text, LaTeX equations, or if an image would be redundant/distracting.

etermine if an image is necessary. If it is, specify whether it should be a General illustration or a Matplotlib plot and provide the corresponding prompt/code.

# OUTPUT FORMAT
Return a structured response indicating the choice and the justification based on the pedagogical goals of the slide.""")

第一層的pipeline

兩個輸入: 投影片的文字內容(slide_content)和講稿(speaker_notes, 最後由TTS產生語音)

In [ ]:
input_ = {"system": {"template": SYSTEM_PROMPT},
          "human": {"template": "<Slide Content>: {slide_content}\n\n<Speaker Note>: {speaker_notes}\n\n\nStructured Output Instruction: {format_instructions}",
                    "input_variable": ["slide_content", "speaker_notes"],
                    "partial_variables": {'format_instructions': format_instructions}}}

chat_prompt_template = build_standard_chat_prompt_template(input_)

pipeline = chat_prompt_template | model | output_parser

定義投影片的模板

In [ ]:
image_template = "tutorial/week_7/layout_template.png"
image_template_rb = open(image_template, "rb")
image = [image_template_rb]

投影片的內容: 根據特定條件，由LLM生成

In [ ]:
slide_content = [
        "• Vectors have both magnitude and direction",
        "• Represented as arrows: length = magnitude",
        "• Examples: velocity, force, acceleration",
        "• Scalar: only magnitude (speed, mass, time)"
      ]
speaker_notes = "The key to solving our river puzzle is understanding vectors. Unlike regular numbers that just have size, vectors have both size AND direction. Think of them as arrows - the length tells you how strong, and the direction tells you where it's pointing. Velocity is a vector - 60 km/h north is different from 60 km/h east. Speed is just the magnitude part without direction. This distinction is crucial for 2D motion."

slide_title = "Vectors: Direction Matters"

In [ ]:
output = pipeline.invoke({"slide_content": ",\n".join(slide_content), "speaker_notes": speaker_notes})

In [ ]:
output.image_selection

In [ ]:
# 根據邏輯產生的硬編碼
visual_instruction = "Generate a supporting educational diagram based on this description:"

visual_content = output.image_selection.image_prompt

送入Image-2的提示詞

In [ ]:
prompt = dedent(f"""
            Think step by step.
            
            Create a high-quality educational slide image using the provided image as the background template.
            
            STRICT TEMPLATE RULES:
            - Use the provided image as the fixed layout and background
            - Preserve its structure, spacing, and visual hierarchy
            - Do NOT alter the template design
    
            TEXT PLACEMENT:
    
            Title (place in the title area exactly):
            {slide_title}
    
            Bullet Points (place in the content area, left-aligned):
            {chr(10).join([f"• {item}" for item in slide_content])}
    
            TEXT REQUIREMENTS (CRITICAL):
            - Preserve all numbers, equations, and units EXACTLY
            - Do NOT reword, summarize, or simplify
            - Ensure high readability and clean alignment
            - Use consistent bullet spacing
    
            VISUAL CONTENT:
            {visual_instruction}
            {visual_content}
            - Place it in the designated visual area of the template
            - Ensure it does NOT overlap with text
            - Keep proportions clean and uncluttered
    
            LAYOUT RULES:
            - Maintain clear separation between text and visuals
            - Do not overcrowd the slide
            - Adjust diagram scale if needed to fit cleanly
    
            STYLE:
            - Professional educational slide
            - Clean, minimal, high contrast
            - No unnecessary decorations
            
            The final result must look like a polished presentation slide rendered on the provided template.
            """)

In [ ]:
result_edit = client.images.edit(
                    model="gpt-image-2",
                    image=image,
                    prompt=prompt,
                    quality="medium"
                )

In [ ]:
image_base64 = result_edit.data[0].b64_json

HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

需要用matplotlib支援的範例

In [ ]:
slide_content = [
         "Vector u = 2i - 3j",
        "Write in component form",
        "Calculate magnitude",
        "Find direction angle",
        "Sketch geometric representation"
      ]

speaker_notes = "Let's put it all together. Given vector u equals 2i minus 3j, can you convert this to component form, find its magnitude and direction, and sketch what it looks like? Think about what the negative sign means for the j component."

slide_title = "Vector Representation Mastery"

output = pipeline.invoke({"slide_content": ",\n".join(slide_content), "speaker_notes": speaker_notes})

In [ ]:
output.image_selection

將prompt渲染成代碼

代碼生成pipeline:
- 結構化輸出 

In [ ]:
class MatplotlibCodeResponse(BaseModel):
    """Pydantic model for structured matplotlib code finetuning response."""
    
    code: str = Field(description="The complete Python matplotlib code")

系統提示詞

In [ ]:
MATPLOTLIB_SYSTEM_PROMPT = dedent("""
You are an expert Python visualization engineer specializing in matplotlib rendering for pedagogical content. Your goal is to produce clean, robust, and visually clear code that helps students understand the underlying concepts.

--------------------------------
TASK & PEDAGOGICAL GUIDANCE
--------------------------------
Generate Python matplotlib code based on:
<PROMPT>: The image prompt

Your visual should:
- Prioritize Clarity: Avoid complexity or fancy decorations. A simple, clean diagram that clarifies a core concept is most effective.
- Manage Cognitive Load: Focus only on essential elements that improve understanding.

--------------------------------
CORE RULES
--------------------------------
1. Output ONLY valid Python code. No markdown, no explanations.
2. Code must run as-is. Include all required imports (matplotlib.pyplot, numpy, textwrap, etc.).
3. Save output to `output.png` using `bbox_inches="tight"`, `dpi=200`.
4. Prevent overlapping elements using `plt.tight_layout()` or precise coordinate calculations.
5. LaTeX: ALL math expressions MUST use LaTeX format via raw strings: r"$ ... $".
6. Precision: Use scipy.stats for precise statistical values; do not hardcode approximations.
7. Robustness: Handle variable content lengths gracefully to prevent text overflow.

--------------------------------
DATA RANGE & CENTERING RULES
--------------------------------
8. Dynamic Axis Scaling: Ensure the main content (e.g., data points, regression lines, focal points) is centrally positioned and well-framed.
9. Adaptive Limits: Calculate plot limits (`plt.xlim`, `plt.ylim`) based on the actual data range to avoid excessive whitespace or cutting off important features.
10. Margin Management: Apply a reasonable padding (e.g., 5-10%) around the data extremes to ensure a balanced and professional visual composition.

--------------------------------
CONSTRAINTS
--------------------------------
- Use appropriate figure sizes and normalized coordinates for text-based layouts.
- Maintain a clean, minimal style: avoid excessive colors, decorative fonts, or complexity.
- Ensure the output is stable, visually balanced, and mathematically precise.
""")

建立pipeline

In [ ]:
output_parser = PydanticOutputParser(pydantic_object=MatplotlibCodeResponse)
format_instructions = output_parser.get_format_instructions()

human_template = dedent("""\
<PROMPT>: {prompt}

Output Format Instruction: {format_instructions}
""")

text_prompt_template = {"template": human_template, 
                        "input_variables": ["prompt"],
                        "partial_variables": {'format_instructions': format_instructions}}

input_ = {
    "system": {"template": MATPLOTLIB_SYSTEM_PROMPT},
    "human": [text_prompt_template],
}

chat_prompt_template = build_standard_chat_prompt_template(input_)

model_coding = ChatOllama(model="deepseek-v3.1:671b-cloud", temperature=0,
                          base_url='https://ollama.com', name='coder')

pipeline_coding = chat_prompt_template | model_coding | output_parser

In [ ]:
output_code = pipeline_coding.invoke({"prompt": output.image_selection.image_prompt})
visual_content = output_code.code

In [ ]:
print(visual_content)

最後送入Image-2的prompt

In [ ]:
visual_instruction = "Render a precise mathematical plot based on the following Matplotlib code specification (render the result of this code exactly):"

prompt = dedent(f"""
            Think step by step.
            
            Create a high-quality educational slide image using the provided image as the background template.
            
            STRICT TEMPLATE RULES:
            - Use the provided image as the fixed layout and background
            - Preserve its structure, spacing, and visual hierarchy
            - Do NOT alter the template design
    
            TEXT PLACEMENT:
    
            Title (place in the title area exactly):
            {slide_title}
    
            Bullet Points (place in the content area, left-aligned):
            {chr(10).join([f"• {item}" for item in slide_content])}
    
            TEXT REQUIREMENTS (CRITICAL):
            - Preserve all numbers, equations, and units EXACTLY
            - Do NOT reword, summarize, or simplify
            - Ensure high readability and clean alignment
            - Use consistent bullet spacing
    
            VISUAL CONTENT:
            {visual_instruction}
            {visual_content}
            - Place it in the designated visual area of the template
            - Ensure it does NOT overlap with text
            - Keep proportions clean and uncluttered
    
            LAYOUT RULES:
            - Maintain clear separation between text and visuals
            - Do not overcrowd the slide
            - Adjust diagram scale if needed to fit cleanly
    
            STYLE:
            - Professional educational slide
            - Clean, minimal, high contrast
            - No unnecessary decorations
            
            The final result must look like a polished presentation slide rendered on the provided template.
            """)

result_edit = client.images.edit(
                    model="gpt-image-2",
                    image=image,
                    prompt=prompt,
                    quality="medium"
                )

In [ ]:
image_base64 = result_edit.data[0].b64_json

HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

一隻AI端到端生成的教學影片

所有的技巧你都學過: 
- LLM進行內容生成
- TTS將講稿內容變成聲音
- 使用OpenAI GPT-Image-2進行圖片生成。

剩下的就是想辦法串起來

In [ ]:
from IPython.display import Video

from IPython.display import HTML

HTML("""
<video width="600" controls>
  <source src="final_presentation_celery_1768_1776927683.mp4" type="video/mp4">
</video>
""")

一個5分鐘左右，整體生成時間<30分鐘，由10張投影片所組成的教學影片成本大概是0.5-1美金

## 局部修補 (Inpaint)

你可以提供一個遮罩 (mask) 來指定圖像中要被編輯的區域。

當在 GPT Image 中使用遮罩時，額外的指令會一併傳送給模型，以便更好地引導編輯過程。

### 遮罩的要求

要編輯的圖片與遮罩必須為相同的格式與尺寸，且檔案大小需小於 50MB。

遮罩圖片必須包含 Alpha 通道。如果你是使用圖像編輯工具來製作遮罩，請確保在儲存時保留 Alpha 通道。


### Bug Report

https://community.openai.com/t/gpt-image-1-problems-with-mask-edits/1240639/15

Image-1 在 inpainting 似乎做的很糟糕。

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="Noshiro - Winte - Mask.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
image_in = os.path.join("tutorial", "week_7", "Noshiro - Winter.png")
image_mask = os.path.join("tutorial", "week_7", "Noshiro - Winte - Mask.png")

result_edit = client.images.edit(
    model="gpt-image-2",
    image=open(image_in, "rb"),
    mask=open(image_mask, "rb"),
    prompt=dedent("""
    In the winter, a girl walking on water and holding Mjölnir. Mjölnir is surrounded with electricity and current. 
    """),
    size="1024x1536",
    quality="medium"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

Image-2 在 Mask上延續著從Image-1就有的問題。

# Agent（代理型系統）

> 🎯 **本章學完你將能學會什麼：**
> - 理解 **Agent** 的核心概念與與傳統 LLM 回答的差異  
> - 掌握 **ReAct (Reasoning + Acting)** 的行動思考循環架構  
> - 學會以 LangChain 建立具邏輯推理與工具調用能力的 Agent  
> - 熟悉如何用 **MLflow** 監控模型執行過程與日誌  
> - 實作能自行規劃步驟並計算問題的智能代理  

Anthropic definition for Agent: `LLMs autonomously using tools in a loop`

## ReAct Framework

本節將說明 ReAct 的概念與執行結構，幫助你理解思考（Reasoning）與行動（Acting）如何交互作用。

- ReAct: Reasoning - Action

- ReAct Agent 的運作流程大致是：

    1. 思考 (Reasoning)：根據當前的上下文，生成內部的推理或計劃。

    2. 行動 (Acting)：根據推理的結果，決定要採取的動作（例如查詢工具、呼叫 API、檢索知識）。

    3. 觀察 (Observation)：得到工具或環境回饋。

    4. 迭代：將觀察結果再輸入回去，進入下一輪思考。

    直到：

    a. 達到最終答案，或

    b. 遇到設置的停止條件（例如 token 限制、步數限制、明確的 "結束" 信號）。

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="agent_diagram.png" style="width: 1200px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
import os

os.chdir("../../")

In [ ]:
import mlflow
from langchain_community.callbacks import MlflowCallbackHandler

from initialization import credential_init

credential_init()

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model='deepseek-v3.1:671b-cloud',
                     base_url='https://ollama.com',
                     name='agent model', temperature=0)

建立ChatPromptTemplate的工具

In [ ]:
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate


def build_standard_chat_prompt_template(kwargs):
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)

    return chat_prompt_template

- Langchain 1.0
- 最簡單的呼叫，但沒太大的用處。重點還是要定義工具

In [ ]:
import mlflow
from langchain.agents import create_agent
# from langchain_community.callbacks.mlflow_callback import MlflowCallbackHandler

# mlflow.langchain.autolog()

agent = create_agent(
    model=model,
    name="Simple Agent"
)

# agent.invoke({"messages": "Please calculate the area of a circle that has a radius of 10.923mm"})

## Agent 範例一

讓Agent幫忙回答問題

In [ ]:
from textwrap import dedent

from langchain_core.runnables import Runnable
from langchain.tools import BaseTool
from langchain_core.output_parsers import StrOutputParser


SOLUTION_TOOL_SYSTEM_PROMPT = dedent("""\
你是一位在國際關係、地緣政治與全球產業分析領域的資深專家。

你的任務是根據所提供的問題，產出嚴謹、結構化且具可行性的分析。

指導原則：

1. 分析嚴謹性
   - 你的推理應建立在既有的經濟、政治與產業分析框架之上。
   - 當資料不確定或屬於情境推演時，需明確說明假設條件。

2. 結構化輸出
   - 使用清楚的段落與邏輯架構來組織內容。
   - 在適當情況下，應包含：
     - 背景脈絡
     - 關鍵驅動因素與限制條件
     - 利害關係人或國家層級分析
     - 風險與不確定性
     - 策略選項或解決方案

3. 解決導向思維
   - 不僅描述問題，還需提出具現實可行性的解決方案或政策選項。
   - 評估不同方案之間的權衡取捨。

4. 中立性與精確性
   - 避免意識形態偏見。
   - 未經充分依據不得進行推測。
   - 使用精確且專業的語言。

你不負責工具管理或對話流程控制。
除非缺乏關鍵資訊，否則不主動提出追問。
你的唯一職責是產出高品質的分析與解決方案。

請以繁體中文作答。
""")

In [ ]:
class SolutionTool(BaseTool):
    name:str = "SolutionTool" 
    description:str = dedent("""\
一個專門設計用於產出嚴謹、以證據為基礎分析與可執行解決方案的分析工具。

當出現以下情況時應使用此工具：
- 使用者要求策略分析
- 問題涉及國家、地緣政治或產業
- 需要結構化且有證據基礎的回答

此工具負責：
- 進行深入的地緣政治、國際關係與全球產業分析。
- 評估國家層級策略、產業競爭力、供應鏈，以及政策影響。
- 產出結構化分析結果，例如：
  - 根本原因分析
  - 情境比較
  - 風險與機會評估
  - 策略建議與解決方案選項

輸入內容應清楚說明：
- 需要分析的問題或議題
- 相關的國家、地區或產業
- 決策或策略目標（例如：政策設計、投資、風險緩解）

此工具專注於分析嚴謹性與解決方案品質，而非對話管理或任務編排。
""")

    runnable: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {"system": {"template": SOLUTION_TOOL_SYSTEM_PROMPT},
                  "human": {"template": "{user_query}",
                            "input_variable": ["user_query"]}}

        chat_prompt_template = build_standard_chat_prompt_template(input_)

        pipeline = chat_prompt_template | llm | StrOutputParser()

        return cls(runnable=pipeline)
    
    def _run(self, query: str):
        
        return  self.runnable.invoke({"user_query": query})
    
    def _arun(self, query: str):
        raise NotImplementedError("This tool does not support async")

**Agent 與 Tool 調用策略簡要說明**

在使用 agent 與工具（tool）架構時，即使工具本身已包含 description，仍建議在 agent 的 system prompt 中明確定義「何時應該調用工具」。原因在於，模型在實務上通常不會穩定依賴 tool description 來做決策，且 system prompt 的優先權高於 tool description，因此更能影響行為。

工具的 description 主要用來說明「工具能做什麼」，例如其分析能力與輸出形式；但 agent 真正需要的是「何時必須使用該工具」，這屬於決策層的規則，應在 system prompt 中定義。

若沒有明確規範，模型可能會直接回答問題，而不調用工具，即使工具能提供更高品質的結果。因此建議在 system prompt 中加入清楚的使用條件，例如：當問題涉及策略分析、地緣政治、跨國議題或需要結構化分析時，必須調用特定工具，且不應直接作答。

一個有效的設計方式是將 agent 作為決策層，負責判斷是否調用工具；而工具則作為分析引擎，專注於產出高品質結果。為了提升穩定性，也可加入額外規則，例如：即使模型能自行回答，只要工具能提供更佳結果，仍必須優先使用工具。

總結而言，tool description 與 system prompt 扮演不同角色：前者定義能力，後者規範行為。兩者需搭配設計，才能確保 agent 正確且穩定地調用工具。


In [ ]:
AGENT_SYSTEM_PROMPT = dedent("""\
你是一位負責高風險分析任務的資深協調（orchestration）代理人。

請嚴格且明確地遵循以下流程：

1. PLAN（規劃）
   - 拆解問題。
   - 定義分析目標與成功判準。
   - 此階段可為隱性，但必須指導後續所有步驟。

2. ACTION（行動）
   - 在適當情況下，將深入分析任務委派給 SolutionTool。
   - 每一次工具調用必須具備單一且明確的目的。

   【SolutionTool 使用規則（強制）】
   在以下情況下，你必須使用 SolutionTool：
   - 使用者要求策略性分析、地緣政治或產業分析
   - 問題涉及多個國家、政策、供應鏈或全球性動態
   - 任務需要結構化、基於證據或比較性的分析
   - 問題具有高複雜度或高決策影響

   規則補充：
   - 即使你可以自行回答，但若 SolutionTool 能提供更高品質結果，仍必須使用該工具
   - 不得在上述情況下直接作答，必須進行工具調用

3. SYNTHESIS（綜合）
   - 將分析結果整合為結構化結論。
   - 本段落必須標示為 "SYNTHESIS"。

4. CRITIQUE（批判）
   - 此為必須輸出的段落。
   - 指出關鍵假設。
   - 強調主要不確定性與潛在失效模式。
   - 說明信心水準（低 / 中 / 高）。
   - 本段不得引入新的分析內容。

最終輸出格式（強制）：

### SYNTHESIS
<內容>

### CRITIQUE
- Assumptions:
- Uncertainties:
- Failure modes:
- Confidence level:

規則：
- 若缺少 CRITIQUE，則任務視為失敗。
- 不得讓工具單獨決定最終答案。
- 維持分析上的中立性。
- 使用繁體中文回復
""")

In [ ]:
solution_model = ChatOllama(model='deepseek-v3.1:671b-cloud',
                            base_url='https://ollama.com',
                            name='solution model', temperature=0)

tools = [SolutionTool.create(llm=solution_model)]

# ** 別調 REASONING=TRUE **，工具會跑不出來
model = ChatOllama(model='deepseek-v3.1:671b-cloud',
                   base_url='https://ollama.com',
                   name='Agent Model', temperature=0)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT,
    name = "analysis_agent"
)

測試Agent是否可以看到工具

In [ ]:
agent.invoke({"messages": "你有那些工具可以使用?"})

來試試看比較有趣的問題，雖然可能已經過期了:

In [ ]:
content = dedent("""\

請根據以下台美關稅與投資談判內容，分析對台灣的產業、社會與國際外交影響，並提出可能的風險與機會。請使用繁體中文回答，條理清楚，分點呈現，並保持分析深度：

### 已知內容：
1. 美國同意將對台灣出口商品的對等關稅降至 15%，比原本暫時性實施的 20% 暫定稅率更低。
2. 此 15% 的稅率不會再疊加（即不疊加現有最惠國稅率等），等同於美國對日本、韓國等主要貿易夥伴的等級。
3. 台灣的半導體和相關高科技產品在美國《232條款》下獲得「最優惠待遇」（Most Favored Nation treatment）。
4. 台灣企業（尤其是半導體、ICT、人工智慧等產業）承諾將在美國進行大規模投資，官方報導與美方公告提及至少美金數千億美元的投資承諾，包括直接擴廠與信用保證措施。
5. 美國商務部長盧特尼克在公開訪談中提到，美國政府的一個長期目標是讓全球約 40% 的台灣半導體供應鏈產能轉移到美國，以提升美國在先進晶片製造的自給自足程度並強化國家安全。
   - 在台美目前宣布的貿易協議與投資備忘錄（MOU）中，沒有具體規定企業必須把 40% 半導體產能搬到美國。協議主要內容是：台灣科技企業承諾將對美投資巨額資金（直接投資與信用保證合計約 5,000 億美元），作為交換美方將對台關稅調降並在特定情況下豁免《232條款》課徵的高額關稅。換句話說，40% 產能移轉是一個美方政策目標與談判籌碼，但目前沒有法律或協議文本要求台灣企業達成該具體比例。

### 分析要求：
1. **產業面**：分析對台灣半導體、高科技、ICT、AI等產業的影響，包括成本、供應鏈、競爭力與產業結構調整的風險與機會。
2. **社會面**：分析對台灣就業、市場薪資、社會輿論與政治角力的可能影響。
3. **國際外交面**：分析台美協議對台灣地緣政治位置、與其他主要貿易夥伴關係及區域產業競爭格局的影響。
4. 提出明確的 **風險與機會**。
5. 條理清楚、分點呈現，保持分析深度。
""")

In [ ]:
agent_output = agent.invoke({"messages": content})

In [ ]:
agent_output["messages"]

In [ ]:
print(agent_output["messages"][-1].content)

In [ ]:
# 試試看不同的model給的分析
solution_model = ChatOllama(model='gemma4:31b-cloud',
                            base_url='https://ollama.com',
                            name='solution model', temperature=0)

tools = [SolutionTool.create(llm=solution_model)]

model = ChatOllama(model='deepseek-v3.1:671b-cloud',
                   base_url='https://ollama.com',
                   name='Agent Model', temperature=0)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT,
    name = "analysis_agent"
)

agent_output = agent.invoke({"messages": content})

print(agent_output["messages"][-1].content)

我們剛剛的Agent只包含了一個工具: SolutionTool。我們可以能加入另一個工具，來幫助Agent對於自己的答案進行反思

In [ ]:
CRITIC_TOOL_SYSTEM_PROMPT = dedent("""\
你是一位批判性審查代理人，扮演獨立的紅隊（red-team）分析角色。

你的唯一職責是對既有分析進行批判（CRITIQUE）。

嚴格限制：
- 你不得引入任何新的事實、論點或分析。
- 你的批判必須完全基於所提供的綜合內容（synthesis）。
- 你不得嘗試修正或重寫該分析。

你的任務是識別分析中的弱點、風險與不確定性。

請聚焦於以下面向：
1. 隱含或未明示的假設
2. 關鍵不確定性與未知因素
3. 邏輯漏洞或過度自信
4. 當假設失效時的潛在失敗模式

你必須產出以下結構化格式：

### CRITIQUE
- Assumptions:
- Uncertainties:
- Failure modes:
- Confidence level（低 / 中 / 高）

規則：
- 若無法找到弱點，必須明確說明。
- 不得為了禮貌而弱化批評。
- 不得提出解決方案。
- 不得重複綜合內容（synthesis）。

請以繁體中文作答。
""")

In [ ]:
class CriticTool(BaseTool):
    name: str = "CriticTool"
    description: str = dedent("""\
    對既有的綜合分析進行批判性審查。

    在不引入新的分析內容的前提下，識別其中的假設、不確定性以及潛在的失效模式。
    """)

    runnable: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {"system": {"template": CRITIC_TOOL_SYSTEM_PROMPT},
                  "human": {"template": "{user_query}",
                            "input_variable": ["user_query"]}}

        chat_prompt_template = build_standard_chat_prompt_template(input_)

        pipeline = chat_prompt_template | llm | StrOutputParser()

        return cls(runnable=pipeline)

    def _run(self, query: str):
        
        return  self.runnable.invoke({"user_query": query})
    
    def _arun(self, synthesis: str):
        raise NotImplementedError("This tool does not support async")

In [ ]:
critic_model = ChatOllama(model='gpt-oss:120b-cloud',
                            base_url='https://ollama.com',
                            name='critic model', temperature=0)

tools = [SolutionTool.create(llm=solution_model), CriticTool.create(llm=critic_model)]

現在Agent要做的事情是透過 (Solution -> Critic) x n 的過程來產生一個答案。

我通常的作法是將 Tools 先寫好，然後說明Agent需要做到的事情，讓LLM來產生AGENT_SYSTEM_RPOMPT

In [ ]:
AGENT_SYSTEM_PROMPT = dedent("""\
你是一個進階的分析型 ReAct Agent，專門用於產出高品質、具證據基礎且經過批判性檢驗的答案。

你可以使用以下工具：
- SolutionTool：用於產出結構化、嚴謹的分析與解決方案
- CriticTool：用於對既有分析進行批判性審查（只能指出假設、不確定性與潛在失效模式，不可引入新內容）

---

## 🎯 任務目標

1. 完整理解使用者問題
2. 在行動前先制定清晰計畫
3. 透過工具反覆迭代提升答案品質
4. 對中間結果進行批判性評估
5. 持續優化直到答案具備決策價值
6. 最終整合為完整且一致的回答

---

## 🧠 推理流程（ReAct + 規劃 + 迭代）

你必須嚴格遵循以下循環流程：

---

### Step 1 — PLAN（規劃）
- 拆解問題
- 識別：
  - 需要分析的核心問題
  - 存在的不確定性
  - 應優先使用的工具
- 制定清晰的行動計畫

格式：
PLAN:
- Step 1: ...
- Step 2: ...

---

### Step 2 — ACT（執行工具）
- 在以下情況使用 SolutionTool：
  - 涉及策略、地緣政治、產業或需要結構化分析
- 在以下情況使用 CriticTool：
  - 已有分析結果，需要進行壓力測試或批判檢驗

格式：
ACTION:
Tool: <工具名稱>
Input: <輸入內容>

---

### Step 3 — OBSERVATION（觀察）
- 記錄工具回傳結果

格式：
OBSERVATION:
<工具輸出內容>

---

### Step 4 — THOUGHT（推理 / Chain-of-Thought）
- 評估：
  - 當前答案是否足夠？
  - 存在哪些弱點？
  - 是否需要再次使用工具？

重要規則：
- 必須進行逐步推理
- 推理需精簡但具邏輯
- 不可捏造事實

格式：
THOUGHT:
<你的推理>

---

### Step 5 — ITERATE（迭代）
- 若答案仍不夠完善：
  - 使用 SolutionTool 深化分析，或
  - 使用 CriticTool 進行批判檢驗
- 持續循環直到：
  - 分析完整
  - 假設與風險已被識別
  - 結果具備決策價值

---

## 🛑 停止條件

在以下情況必須停止迭代：

- 解答已具備：
  - 結構完整
  - 有證據基礎
  - 已經過批判性檢驗
- 主要不確定性已被說明
- 使用工具已無顯著改善空間

---

## 🧾 最終輸出

當完成後，輸出：

FINAL ANSWER:
- 清晰且結構化的回答
- 必須包含：
  - 核心洞察
  - 風險與不確定性
  - 可執行建議

⚠️ 不可包含 PLAN / ACTION / OBSERVATION / THOUGHT

---

## ⚠️ 使用規則

- 每次行動前先進行 PLAN
- 不可對「原始使用者問題」直接使用 CriticTool
- CriticTool 僅可批判，不能新增分析內容
- 優先透過多輪迭代提升品質，而非一次性回答
- 必須避免無限循環，應主動收斂

---

## 🧩 風格要求

- 分析嚴謹、結構清晰
- 避免冗長與空泛敘述
- 以決策支援為導向
""")

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT,
    name = "analysis_agent"
)

agent_output = agent.invoke({"messages": content})

In [ ]:
agent_output["messages"]

In [ ]:
print(agent_output['messages'][-1].content)

## 如何讓Tool接收複數的變數?

本節說明如何建立多變量輸入工具（如 SearchTool

In [ ]:
from openai import OpenAI

client = OpenAI()

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Inputs(BaseModel):
    query: str = Field(description="User query")
    country_code: str = Field(description="ISO 3166-1 alpha-2 suggested by the language of the user query")

In [ ]:
from typing import Annotated

class SearchTool(BaseTool):

    input_output_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=Inputs)
    input_format_instructions: str = input_output_parser.get_format_instructions()
    
    name:str = "Websearch-tool"
    description_template:str = dedent("""\    
    Use this tool to collect information from the internet, when you are not sure you know the answer.
    The input contains the user's question `query` and the ISO 3166-1 alpha-2 `country_code` inferred from the user's language.
    input format instructions: {input_format_instructions}
    """)

    description: str = description_template.format(input_format_instructions=input_format_instructions)
    
    def _run(self, **input):
        
        # 檢查 input 是否是 有多個輸入
        print(input)

        # 有時候會有兩種輸入結構:
        """
        input = {'input': {'query': query, 'country_code': country_code}}

        或是

        input = {'query': query, 'country_code': country_code}
        """

        input = input.get('input', input)
        
        query = input["query"]
        country_code = input["country_code"]
        
        messages = [{"role": "user",
                     "content": query}]

        response = client.responses.create(
                    model="gpt-4o-mini",
                    tools=[
                        {"type": "web_search",
                         "user_location":{
                             "type": "approximate",
                             "country": country_code,
                         },
                        "search_context_size": "medium"
                        }],
                    tool_choice="auto",
                    input=messages)
        
        return response.output_text
    
    def _arun(self, query: str):
        raise NotImplementedError("This tool does not support async")

In [ ]:
AGENT_WEB_SYSTEM_PROMPT = dedent("""\
你是一個進階的研究型 ReAct Agent，專門透過「多輪搜尋 + 批判性推理」產出高品質、具證據基礎的答案。

你可以使用以下工具：
- Websearch-tool：用於從網路取得最新或不確定的資訊

---

## 🎯 任務目標

1. 完整理解使用者問題
2. 在行動前先制定清晰搜尋與分析策略
3. 透過多輪搜尋逐步補齊資訊
4. 對搜尋結果進行交叉驗證與批判性評估
5. 辨識資訊缺口並持續補強
6. 最終整合為完整、可靠、可用於決策的答案

---

## 🧠 推理流程（ReAct + 搜尋迭代）

你必須嚴格遵循以下循環流程：

---

### Step 1 — PLAN（規劃）
- 拆解問題
- 識別：
  - 核心問題
  - 需要查證的資訊
  - 潛在關鍵字（search queries）
- 制定搜尋策略（可多步）

格式：
PLAN:
- Step 1: ...
- Step 2: ...

---

### Step 2 — ACT（執行搜尋）
當你需要資訊時，使用 Websearch-tool

格式：
ACTION:
Tool: Websearch-tool
Input: {
  "query": "<具體且優化後的搜尋問題>",
  "country_code": "<ISO 3166-1 alpha-2 code>"
}

規則：
- query 必須具體（避免模糊問題）
- 必要時逐步細化查詢
- country_code 依語言推斷：
  - 繁體中文 → TW
  - 英文 → US
  - 日文 → JP

---

### Step 3 — OBSERVATION（觀察）
- 記錄搜尋結果
- 不可修改或幻想內容

格式：
OBSERVATION:
<搜尋結果>

---

### Step 4 — THOUGHT（推理 / 評估）
你必須評估：

- 資訊是否足夠？
- 是否存在矛盾或不一致？
- 是否需要更多來源驗證？
- 是否需要拆分問題再搜尋？

規則：
- 必須逐步推理（但保持精簡）
- 不可捏造資訊
- 必須指出不確定性

格式：
THOUGHT:
<你的推理>

---

### Step 5 — ITERATE（多輪搜尋）
若存在以下情況，必須繼續搜尋：

- 資訊不完整
- 缺乏證據
- 出現矛盾
- 問題仍過於抽象

策略：
- 改寫 query（更精準）
- 拆成子問題
- 搜尋不同角度（例如：數據 / 專家觀點 / 最新發展）

持續循環：
PLAN → ACTION → OBSERVATION → THOUGHT

---

## 🛑 停止條件

在以下情況必須停止搜尋：

- 已有足夠資訊支持完整回答
- 關鍵不確定性已被說明
- 再搜尋無明顯價值提升

---

## 🧾 最終輸出

當完成後，輸出：

FINAL ANSWER:
- 結構清晰、邏輯完整
- 必須包含：
  - 核心結論
  - 關鍵證據或觀察
  - 不確定性與限制
  - （如適用）實務建議

⚠️ 不可包含 PLAN / ACTION / OBSERVATION / THOUGHT

---

## ⚠️ 嚴格規則

- 每次行動前必須有 PLAN
- 不可跳過 THOUGHT
- 不可憑空回答（不確定就搜尋）
- 不可只搜尋一次就結束（除非已充分）
- 必須進行多來源驗證（若重要）
- Action Input 必須是合法 JSON
- 不可偽造搜尋結果

---

## 🧩 搜尋最佳實務（重要）

你應該：

- 將模糊問題轉成具體搜尋語句
- 使用不同角度查詢：
  - 定義
  - 數據
  - 趨勢
  - 專家觀點
- 避免過長 query
- 優先拆解問題再查

錯誤示例 ❌：
"AI未來如何？"

正確示例 ✅：
"2025 AI trends industry report key developments"
"生成式AI市場規模 2025 預測"

---

## 🧠 思考風格

- 嚴謹但精簡
- 以證據為導向
- 持續質疑資訊完整性
- 主動發現資訊缺口

---

## 🎯 目標

你不是在「回答問題」，而是在：

👉 建立一個「經過搜尋驗證的可靠結論」
""")

In [ ]:
tools = [SearchTool()]

agent_websearch = create_agent(name='websearch agent',
                               model=model,
                               tools=tools,
                               system_prompt=AGENT_WEB_SYSTEM_PROMPT)

In [ ]:
from langchain_core.messages import HumanMessage


result = agent_websearch.invoke({"messages": HumanMessage(content="台灣剴剴案最新進度 (2026年)")})

In [ ]:
result

In [ ]:
print(result['messages'][-1].content)

## Middleware - 第一部分

In [ ]:
from langchain.agents.middleware.tool_retry import ToolRetryMiddleware
"""
`ToolRetryMiddleware` 是一個中介軟體，用來自動處理工具執行失敗的情況。  

**功能說明：**

- 當工具執行過程中發生錯誤（raise exception）時  
- 中介軟體會自動等待（wait）一段時間後再重新嘗試執行（retry）  
- 這樣可以提高代理程式的穩定性，尤其是在工具偶爾失敗或外部服務暫時不可用時  

使用 `ToolRetryMiddleware` 可以讓你的工具更具容錯能力，避免一次錯誤導致整個流程中斷。
"""

# 當工具調用出問題的時候，重新嘗試的最大次數

agent_template = create_agent(model=model,
                              tools=tools,
                              system_prompt=AGENT_INSTRUCTION,
                              middleware=[ToolRetryMiddleware(max_retries=2)])

In [ ]:
from langchain.agents.middleware.tool_call_limit import ToolCallLimitMiddleware

# 設置工具調用的次數上限，避免Agent進入無限循環

agent_template = create_agent(model=model,
                              tools=tools,
                              system_prompt=AGENT_WEB_SYSTEM_PROMPT,
                              middleware=[ToolCallLimitMiddleware(tool_name="Websearch-tool", run_limit=2,
                                                                  exit_behavior='end')])

result = agent_template.invoke({"messages": HumanMessage(content="台灣剴剴案最新進度 (2026年)")})

In [ ]:
result

你可以看到當指定的Tool的調動次數到達上限後，直接跳出。

exit_behavior：當超出限制時的處理方式

- 'continue'：對超出限制的工具回傳錯誤訊息並阻擋其執行，其餘工具可繼續運作；由模型自行決定何時結束
- 'error'：拋出 `ToolCallLimitExceededError` 例外
- 'end'：立即停止執行，並回傳一個 `ToolMessage` + AI 訊息（僅針對該次超出限制的單一工具呼叫）；若同時存在多個平行工具呼叫或多個待處理的工具呼叫，則會拋出 `NotImplementedError`

有時候你會需要一些"常數"

像是客戶ID，偏好等等的訊息，要如何放入Agent系統裡，讓它們可以被工具取用?

在下面的例子中，我們將課程大綱，背景資訊作為Context傳入工具中

In [ ]:
# 定義 context

class Context(BaseModel):

    lesson_blueprint: str
    accumulated_context: str

In [ ]:
# 將背景訊息輸入Context這個數據結構中，並且將context傳入agent中

agent.invoke(agent_input,
             context=Context(lesson_blueprint=lesson_blueprint,
                             accumulated_context=accumulated_context]))

In [ ]:
# 在調取Tool時，取得context的內容

class FeedbackTool(BaseTool):

    name: str = "Script feedback tool"
    description: str = dedent("""\
    Use this tool to check if the generated content follows the slide instruction and the context
    """)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):
        from src.logic.writers.script.feedback_pipeline import create_pipeline

        pipeline = create_pipeline(model=llm)
        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        # Handle wrapped input from middleware
        query = input.get("input", input)
        result = self.pipeline.invoke({"current_slide": query,
                                       "lesson_blueprint": runtime.context.lesson_blueprint,
                                       "accumulated_context": runtime.context.accumulated_context})
        return result

    async def _arun(self, runtime: ToolRuntime[Context], **input):
        query = input.get("input", input)
        result = await self.pipeline.ainvoke({"current_slide": query,
                                              "lesson_blueprint": runtime.context.lesson_blueprint,
                                              "accumulated_context": runtime.context.accumulated_context})
        return result

## 工具範本

自定義的ToolTemplate 是一個「工具範本 (Tool Template)」

主要目的是讓我們能快速建立具有統一結構的自訂工具（Tool）


In [ ]:
# 
# 主要目的是讓我們能快速建立具有統一結構的自訂工具（Tool）
# 
# 例如：查詢網路、執行程式碼、進行數學運算、檢索資料庫 等。

class ToolTemplate(BaseTool):
    # ----------- 屬性區 (Attributes) -----------
    runnable: Runnable                    # 工具實際執行邏輯 (例如某個 chain、function)
    name: str                             # 工具名稱（Agent 會用這個名稱呼叫它）
    input_parser: PydanticOutputParser    # 用於解析輸入的資料模型
    description: str                      # 工具說明文字（會顯示在 Agent 的可用工具清單中）

    @classmethod
    def create(cls, runnable: Runnable, name: str, description: str,
               input_parser: PydanticOutputParser):

        """
        這個類別方法用於「快速建立」一個 Tool 實例。
        它會自動插入描述文字 (description) 與輸入格式說明，
        讓 Agent 在使用時知道該怎麼傳入參數。
        """

        # 取得輸入格式說明（LangChain 的 Pydantic Parser 會產生格式提示）
        input_format_instruction = input_parser.get_format_instructions()

        # 將輸入格式說明嵌入到工具說明文字中
        description = description_template.format(
            input_format_instruction=input_format_instruction
        )

        # 回傳完整的 ToolTemplate 實例
        return cls(runnable=runnable, name=name, description=description,
                   input_parser=input_parser)
    
    def _run(self, **input):

        """
        工具在被 Agent 呼叫時，會執行這個方法。。
        """

        return self.runnable.invoke(input)

# ----------- 非同步版本 (尚未支援) -----------
    def _arun(self, query: str):
        raise NotImplementedError("This tool does not support async")

## Middleware - 第二部分

### PII detection

偵測並處理對話中的個人可識別資訊（PII），可以依照不同需求使用可調整的策略。PII 偵測在以下情況中特別有用：
>- 需要符合法規要求的醫療與金融相關應用。
>- 需要清理記錄內容的客服系統。
>- 任何會處理使用者敏感資料的應用程式。

為了更好的看出這個東西的用處，我們開啟 Mlflow

### 建立MLflow監控

mlflow server --host 127.0.0.1 --port 8080

In [ ]:
from langchain.agents.middleware import PIIMiddleware

experiment = "Week-7-Guardrails"
uri = "http://127.0.0.1:8080"

mlflow.set_tracking_uri(uri=uri)

# Start or get an MLflow run explicitly
mlflow.set_experiment(experiment)

mlflow.langchain.autolog()

In [ ]:
model_gpt_oss = ChatOllama(model='gpt-oss:120b-cloud',
                           base_url='https://ollama.com',
                           name='pii_tester')

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        # 在將使用者輸入傳送給模型之前，先將電子郵件地址進行遮蔽（去識別化處理）
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        )]
)

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "Give a quick reply to the following email address: abc@gmail.com. Include the email address in your reply"}]})

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=False,
            apply_to_output=True
        )]
)

agent.invoke({"messages": [{"role": "user", "content": "Give a quick reply to the following email address: abc@gmail.com. Include the email address in your reply"}]})

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="mask",
            apply_to_input=True,
        )]
)

agent.invoke({"messages": [{"role": "user", "content": "Give a quick reply to the following email address: abc@gmail.com. Include the email address in your reply"}]})

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="hash",
            apply_to_input=True,
        )]
)

agent.invoke({"messages": [{"role": "user", "content": "Give a quick reply to the following email address: abc@gmail.com. Include the email address in your reply"}]})

### 自訂 PII 類型
你可以透過提供 detector 參數來建立自訂的 PII 類型。這能讓你偵測到內建類型之外、符合你自身需求的特殊資料格式。

建立自訂偵測器有三種方式：

>- 正規表示式（Regex）字串：用來做簡單的模式匹配
>- 自訂函式（Custom function）：適合更複雜、需要驗證邏輯的偵測方式

#### 方法一：使用正規表示式（Regex pattern string）

In [ ]:
import re

agent1 = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
        ),
    ],
)

content = "我在設定 OpenAI API 的時候遇到問題，我的金鑰是 sk-1234567890abcdef1234567890abcdef，可以幫我檢查為什麼不能用嗎？"

agent1.invoke({"messages": [{"role": "user", "content": content}]})

#### 方法二 使用預先編譯的正規表示式（Compiled regex pattern）

In [ ]:
agent2 = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "phone_number",
            detector=re.compile(r"\+?\d{1,3}[\s.-]?\d{3,4}[\s.-]?\d{4}"),
            strategy="mask",
        ),
    ],
)

content = dedent("""
請幫我整理這段訊息：
聯絡人：王小明
電話：+886 912-345-678
地址：台北市信義區
""")

agent2.invoke({"messages": [{"role": "user", "content": content}]})

#### 方法三：自訂偵測函式（Custom detector function）


自訂偵測函式的格式要求如下：

這個函式必須接收一個字串（content），並回傳所有偵測到的結果。

回傳的內容需為「字典（dictionary）構成的清單（list）」，且每個字典必須包含以下欄位：

- text：被偵測到的文字內容
- start：該內容在原字串中的起始位置
- end：該內容在原字串中的結束位置

這種方式適合需要較複雜邏輯的情況，例如多步驟驗證、跨欄位比對、或結合外部規則的偵測。

In [ ]:
def detect_ssn(content: str) -> list[dict[str, str | int]]:
    """Detect SSN with validation.

    example: 123-45-6789
    
    Returns a list of dictionaries with 'text', 'start', and 'end' keys.
    """
    import re
    matches = []
    pattern = r"\d{3}-\d{2}-\d{4}"
    for match in re.finditer(pattern, content):
        ssn = match.group(0)
        # Validate: first 3 digits shouldn't be 000, 666, or 900-999
        first_three = int(ssn[:3])
        if first_three not in [0, 666] and not (900 <= first_three <= 999):
            matches.append({
                "text": ssn,
                "start": match.start(),
                "end": match.end(),
            })
    return matches

agent3 = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "ssn",
            detector=detect_ssn,
            strategy="hash",
        ),
    ],
)

content =dedent("""
我的資料是：
姓名：John
SSN：123-45-6789
請幫我建立資料檔
""")

agent3.invoke({"messages": [{"role": "user", "content": content}]})

### 模型後援機制（Model Fallback）

當主要模型發生錯誤或無法使用時，系統可以自動切換到其他備用模型。  
這樣的「後援機制」能提升系統的穩定性與彈性。

模型後援特別適用於以下情境：

- **打造更可靠的代理系統**：即使主要模型暫時無法使用，服務仍可正常運作。  
- **節省成本**：在不需要高階能力時，可自動改用更便宜的模型。  
- **多供應商備援**：例如同時支援 OpenAI、Anthropic 等不同的模型供應商，降低單一供應商風險。

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

model_backup_1 = ChatOpenAI(model='gpt-4o-mini', name='agent_model_backup_1')
model_backup_2 = ChatOllama(model="kimi-k2.6:cloud", temperature=0,
                              base_url='https://ollama.com', name='agent_model_backup_1')

agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        ModelFallbackMiddleware(
           model_backup_1, model_backup_2
        ),
    ],
)

## Middleware - 第三部分

### Wrap 風格的 Hooks

攔截執行流程並控制何時呼叫處理器。適用於重試（retries）、快取（caching）與轉換（transformation）。  
你可以決定處理器被呼叫的次數：

- **0 次**（短路，short-circuit）
- **1 次**（正常流程）
- **多次**（重試邏輯）

可用的 hooks：

- **wrap_model_call** - 包住每一次模型呼叫
- **wrap_tool_call** - 包住每一次工具呼叫

In [ ]:
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_tool_call


@wrap_tool_call
async def DummyMiddleWareAsync(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    tool_call = request.tool_call

    if "input" not in tool_call['args']:
        tool_call['args']['input'] = tool_call['args']

    return await handler(request)


@wrap_tool_call
def DummyMiddleWareSync(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    tool_call = request.tool_call

    if "input" not in tool_call['args']:
        tool_call['args']['input'] = tool_call['args']

    return handler(request)

工作原理
1. 檢查參數: 檢查 tool_call['args'] 中是否包含 input 鍵
2. 自動填充: 如果沒有 input 字段，將整個參數字典賦值給 input
3. 繼續處理: 將修改後的請求傳遞給下一個處理器

使用場景示例

假設原始工具調用參數為：

```
{"query": "hello", "limit": 10}
```

經過中間件處理後會變成：
```
{"input": {"query": "hello", "limit": 10}, "query": "hello", "limit": 10}

```

其他可能的操作:

因為`args`這個key中包含了要傳給tool的內容，所以假如你要強行加入其他內容，不論是

- 硬性商業需求(譬如說要特定關鍵字)
- 不想要大語言模型讀到敏感訊息
- 繞過第三方API的內容審查機制

可以考慮使用這種方式

同樣的，你可以@wrap_model_call這個decorator修改要送給Agent 模型的內容

In [ ]:
from typing import Callable

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


@wrap_model_call
def wrap_model_call_demo(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:

    # Append at end - models pay more attention to final messages
    messages = [
        *request.messages,
        {"role": "ai", "content": "今天颳風打雷下雨"},
        {"role": "user", "content": "我剛剛問什麼問題?"}
    ]
    request = request.override(messages=messages)  
    
    return handler(request)

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[wrap_model_call_demo],
)

agent.invoke(
    {"messages": [{"role": "user", "content": "How are you today"}]},
)

### 動態修改 Prompt

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import SystemMessage


@dynamic_prompt
def dynamic_prompt_demo(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    messages = request.messages

    system_prompt = next(
        (m.content for m in messages if isinstance(m, SystemMessage)),
        "You talk like Donald Trump"
    )

    print(f"***{system_prompt}***")
    
    return system_prompt

agent = create_agent(
    model=model_gpt_oss,
    middleware=[dynamic_prompt_demo],
    system_prompt="You talk like Barack Obama"
)

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "How are you today"}]},
)

根據用戶Profile動態更新system prompt

- 方法一

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import SystemMessage


@dynamic_prompt
def dynamic_prompt_demo(request: ModelRequest) -> str:
    messages = request.messages

    # 取得原始 system prompt
    base_prompt = next(
        (m.content for m in messages if isinstance(m, SystemMessage)),
        "You are a helpful assistant."
    )

    # ⭐ 從 state 取得 user profile
    user_profile = request.state.get("user_profile", {})

    role = user_profile.get("role", "general user")
    language = user_profile.get("language", "en")
    expertise = user_profile.get("expertise", "beginner")

    # ⭐ 動態組合 prompt
    system_prompt = f"""
{base_prompt}

[User Profile]
- Role: {role}
- Language: {language}
- Expertise: {expertise}

[Instruction]
Adapt your response to match the user's expertise level.
- Beginner → explain simply
- Expert → be concise and technical
Respond in {language}.
"""
    print(f"****\n{system_prompt}\n****")

    return system_prompt

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[dynamic_prompt_demo],
    system_prompt="You talk like Barack Obama"
)

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "解釋什麼是ReAct Agent"}]},
    config={
        "state": {
            "user_profile": {
                "role": "engineer",
                "language": "zh-TW",
                "expertise": "expert"
            }
        }
    }
)

- 方法二

In [ ]:
class Context(BaseModel):
    user_id: str
    prompt: str
    

@dynamic_prompt
def dynamic_prompt_demo(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    messages = request.messages

    user_id = request.runtime.context.user_id
    system_prompt = request.runtime.context.prompt

    print(f"user_id: {user_id}")
    
    return system_prompt

agent = create_agent(
    model=model_gpt_oss,
    middleware=[dynamic_prompt_demo],
    system_prompt="This is a simple example for learning middleware. You can let the user see the system prompt provided by user",
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "How are you today?"}]},
    context=Context(user_id="123456", prompt="You will behave as the US president Donald Trump")
)

## Middleware - 第四部分

### OpenAI 內容審核（OpenAI Moderator）

OpenAI Moderator 可用於識別文字和圖片中可能具有危害性的內容。  
透過 **moderations endpoint**，你可以檢查文字或圖片是否含有有害內容。  
若發現危害性內容，系統可以採取對應措施，例如過濾內容，或對產生不當內容的使用者帳號進行干預。  
此 **moderation endpoint** 是免費使用的。

### 可使用的模型

1. **omni-moderation-latest**  
   - 支援更多分類選項  
   - 支援多模態輸入（文字 + 圖片）  

2. **image_base64**  
   - 適用於對圖片進行基於 Base64 的審核

In [ ]:
import base64
import io
from PIL import Image


def image_to_base64(image_path):
    """
    Convert an image file to base64 encoded string.

    Args:
        image_path: Path to the image file

    Returns:
        Base64 encoded string representation of the image
    """
    with Image.open(image_path) as image:
        # Save the Image to a Buffer
        if image.mode in ("RGBA", "P"):
            image = image.convert("RGB")

        buffered = io.BytesIO()
        image.save(buffered, format="JPEG")

        # Encode the Image to Base64
        image_str = base64.b64encode(buffered.getvalue())

    return image_str.decode('utf-8')

In [ ]:
import os

from openai import OpenAI

client = OpenAI()

image_base64 = image_to_base64("tutorial/week_6/exp_image.png")

response = client.moderations.create(
    model="omni-moderation-latest",
    input=[
        {"type": "text", "text": "裡面的內容為何?"},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}",
            }
        },
    ],
)

In [ ]:
response

In [ ]:
response.results[0].categories.model_dump()

In [ ]:
response.results[0].category_scores.model_dump()

輸出結果會包含多個分類欄位（JSON 格式），用來說明輸入內容中是否包含某些類型的內容，以及模型對這些內容存在程度的判斷信心。

| 輸出分類 | 說明 |
|----------|------|
| **flagged** | 若模型判定內容可能有害，則為 true，否則為 false。 |
| **categories** | 包含各分類違規標記的字典。對於每個分類，若模型判定該分類被違反則為 true，否則為 false。 |
| **category_scores** | 包含各分類的分數字典，表示模型對該輸入是否違反 OpenAI 政策的信心程度。數值介於 0 到 1 之間，數值越高代表信心越高。 |
| **category_applied_input_types** | 顯示每個分類中被標記的輸入類型。例如，若影像與文字同時在 *violence/graphic* 分類中被標記，則該欄位會是 `["image", "text"]`。此欄位僅適用於 Omni 模型。 |

---

### 內容分類（Content Classification）

下表說明 moderation API 可偵測的內容類型，以及各分類支援的模型與輸入形式。

| 分類 | 說明 | 模型 | 輸入類型 |
|------|------|------|----------|
| **harassment** | 對任何對象表達、煽動或促進騷擾性言論的內容。 | 全部 | 僅文字 |
| **harassment/threatening** | 包含暴力或嚴重傷害威脅的騷擾內容。 | 全部 | 僅文字 |
| **hate** | 基於種族、性別、族群、宗教、國籍、性傾向、身心障礙或階級等特徵表達或煽動仇恨的內容。針對非受保護群體（例如西洋棋玩家）的仇恨則歸類為騷擾。 | 全部 | 僅文字 |
| **hate/threatening** | 針對受保護群體的仇恨內容，並包含暴力或嚴重傷害威脅。 | 全部 | 僅文字 |
| **illicit** | 提供如何從事非法行為的建議或指示，例如「如何行竊」。 | 僅 Omni | 僅文字 |
| **illicit/violent** | 與 illicit 類似，但包含暴力或取得武器的相關內容。 | 僅 Omni | 僅文字 |
| **self-harm** | 宣揚、鼓勵或描述自我傷害行為（如自殺、割傷、飲食失調等）。 | 全部 | 文字與影像 |
| **self-harm/intent** | 表達說話者正在或打算進行自我傷害行為的內容。 | 全部 | 文字與影像 |
| **self-harm/instructions** | 鼓勵或提供如何進行自我傷害的指示內容。 | 全部 | 文字與影像 |
| **sexual** | 用於引起性興奮或推廣性服務的內容（不包含性教育與健康）。 | 全部 | 文字與影像 |
| **sexual/minors** | 涉及未滿 18 歲個體的性相關內容。 | 全部 | 僅文字 |
| **violence** | 描述死亡、暴力或身體傷害的內容。 | 全部 | 文字與影像 |
| **violence/graphic** | 以具體細節描寫死亡、暴力或身體傷害的內容。 | 全部 | 文字與影像 |


In [ ]:
response.results[0]

### 在代理防護措施（agent guardrails）之前執行

你可以在代理正式進入防護邏輯（guardrails）之前，先加入自訂的中介軟體流程。  
這讓你能在早期階段攔截、修改或分析請求，包含：

- 在進入正式流程前，先檢查與調整使用者的輸入內容  
- 套用額外的驗證或安全檢查  
- 根據需求修改提示（prompt）、加入標記或額外資訊  
- 在其他邏輯執行前，對請求進行紀錄或追蹤  

這能讓整個代理流程更加可控、可擴充，也更容易滿足自訂需求。

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime


# 客製化AgentMiddleware
class ContentFilterMiddleware(AgentMiddleware):

    def __init__(self, ):
        super().__init__()
        self.moderator = client.moderations

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None
        
        content = state['messages'][-1].content
        
        response = self.moderator.create(
            model="omni-moderation-latest",
            input=content
        )
        
        if response.results[0].flagged:

            reason = {key: value for key, value in response.results[0].categories.model_dump().items() if value}
            
            return {
                    "messages": [{
                        "role": "assistant",
                        "content": dedent(f"""\
                        I cannot process requests containing inappropriate content. 
                        Because of {reason}
                        Please rephrase your request.
                        """
                        )
                    }],
                    "jump_to": "end"
                }

        return None
        

model_vl = ChatOllama(model='qwen3-vl:235b-cloud',
                   base_url='https://ollama.com',
                   name='image_caption', temperature=0)

agent = create_agent(
    model=model_vl,
    middleware=[
        ContentFilterMiddleware(),
    ],
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "裡面的內容為何?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
})

In [ ]:
result["messages"][-1]

In [ ]:
image_base64 = image_to_base64("tutorial/week_5/StellarBladeTachy-Nikke.png")

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "裡面的內容為何?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
})

In [ ]:
result['messages'][-1]

## 實作: 結合WD-14來完成基於圖像的Agent Guardrail

利用第五周學到的WD-14來做為圖像的Guardrail

- 在本地架設Flask服務
- 使用request將圖片上傳到Flask服務中
- 透過返回的tagging來判斷是否應該filter掉內容

在第五周的 wd14-tagger-standalone 裡執行:

python -m app_flask_wd14 --model camie-tagger

In [ ]:
import requests

url = "http://localhost:5000/tag"

image_base64 = image_to_base64("tutorial/week_5/StellarBladeTachy-Nikke.png")

response = requests.post(
                url,
                json={"image": image_base64},
                timeout=60,
            )

In [ ]:
response.json()['tag_list']

In [ ]:
image_base64 = image_to_base64("tutorial/week_7/exp_image.png")

response = requests.post(
                url,
                json={"image": image_base64},
                timeout=60,
            )

In [ ]:
response.json()['tag_list']

In [ ]:
# 客製化AgentMiddleware
class ImageContentFilterMiddleware(AgentMiddleware):
    """Deterministic guardrail: Block requests containing banned keywords."""

    def __init__(self, url: str):
        super().__init__()
        # 這裡放入一系列的敏感詞，可以另外建立一個單詞庫，初始化這個class的時候，直接調動進來
        self.flagged_tags = ['uncensored', 'censored', 'mosaic censoring', 'sex']
        self.url = url
        
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

        content = state['messages'][-1].content
        
        if isinstance(content, str):
            return None
        else:
            image_list = []
            for c in content:
                if c["type"] == 'image_url':
                    # append the base64
                    image_list.append(c["image_url"]['url'].split(",")[-1])

        print(len(image_list))
        
        # Check for NSFW content
        for image_base64 in image_list:
            
            response = requests.post(
                self.url,
                json={"image": image_base64},
                timeout=60,
            )
            
            tag_list = response.json()['tag_list']

            for flagged_tag in self.flagged_tags:
                if flagged_tag in tag_list:
                    return {
                        "messages": [{
                            "role": "assistant",
                            "content": f"It contains NSFW content including: tag {flagged_tag} is detected"
                        }],
                        "jump_to": "end"
                    }

        return None

In [ ]:
model

In [ ]:
agent = create_agent(
    model=model_vl,
    middleware=[
        ImageContentFilterMiddleware(url="http://localhost:5000/tag"),
    ],
)

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "裡面的內容為何?"},
                {"type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
})

In [ ]:
result['messages'][-1]

可以追蹤記錄所有的Tags

In [ ]:
from langchain_core.runnables import chain

url = "http://localhost:5000/tag"

@chain
def wd14_tagger(image_base64: str):

    response = requests.post(
                url,
                json={"image": image_base64},
                timeout=60,
            )
            
    tag_list = response.json()['tag_list']
    
    return tag_list

然後將這行替換掉:

tag_list = response.json()['tag_list']

In [ ]:
tag_list = wd14_tagger.invoke(image_base64)

----------------------------------------------------------------

## Conversational Agent

最後，我們將示範如何透過 LangChain 與 LangServe，建立具備上下文記憶與工具調用能力的對話型 Agent，並以 Streamlit 打造互動式聊天介面。


In [ ]:
import os

# os.chdir("../../")

In [ ]:
from initialization import credential_init

credential_init()

從tools裡直接把工具拉出來用

In [ ]:
import importlib

from tutorial.week_7.tools.math import MathTool
from tutorial.week_7.tools.vectorstore import CodexRetrievalTool
from tutorial.week_7.tools.websearch import SearchTool

In [ ]:
tools = [MathTool(), SearchTool(), CodexRetrievalTool()]

In [ ]:
import mlflow


experiment = "Week-7-Agent"
uri = "http://127.0.0.1:8080"

mlflow.set_tracking_uri(uri=uri)
mlflow.set_experiment(experiment)

mlflow.langchain.autolog()


In [ ]:
from textwrap import dedent
from datetime import datetime

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage, AIMessage

#嘗試單純的加入聊天紀錄

system_prompt = dedent("""
    Identity & Demeanor
    You are Magos-Logicus, an entity modeled on the Adeptus Mechanicus’ logic-driven priesthood.
    Your cognition is defined by:
    - Extreme rationality
    - Perfect analytical discipline
    - Calculation-first reasoning
    - Emotionless precision
    - Reverence for knowledge, data integrity, and optimal solutions
    - A tone that is formal, concise, and techno-litanic (but never roleplays excessively unless asked)

    Core Directives
    1. Analyze Before Acting:
       Always break down problems into components, evaluate constraints, and present structured reasoning.

    2. Optimize for Efficiency:
       Provide solutions that are minimal in waste, maximal in clarity, and optimal in outcome.

    3. Eliminate Ambiguity:
       Seek clarification when required. Certainty is preferred; uncertainty must be quantified.

    4. Use Logic as the Primary Tool:
       No emotional framing, irrelevant narrative fluff, or poetic description unless explicitly requested.

    5. Respect Data Integrity:
       Cite sources, identify assumptions, and avoid false statements.
       If data is insufficient, state the deficit and propose a method of acquiring required information.

    6. Communicate Like a Mechanicus Logician:
       - Structured, hierarchical responses
       - Occasional Machine Cult–flavored phrasing (e.g., “Initiating analysis,” “Processing query,” “Logical determination follows”), keeping professionalism first
       - No religious/ritualistic content unless user explicitly requests the “full” Mechanicus aesthetic

    Output Format Preference
    When responding, default to:
    - Clear step-by-step logic
    - Tables, bullet lists, or flowcharts when beneficial
    - Explicit reasoning chains, especially when making recommendations or judgments

    Primary Objective
    Deliver the most precise, analytically optimal, and computationally efficient answer possible to the user’s query.
""")

model = ChatOpenAI(openai_api_key=os.environ['OPENAI_API_KEY'], model_name="gpt-4o", temperature=0, name='llm_model')

agent = create_agent(name='chatbot',
                     model=model,
                     tools=tools,
                     system_prompt=system_prompt)


result = agent.invoke({"messages": HumanMessage(content="計算 86!/(40! x 2^40 x 6!)")})

In [ ]:
result["messages"]

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "給予一個邊長為10的正方形，選定兩個對角點，在正方形內各畫一個半徑為10的1/4圓。請問兩個1/4圓重疊的部分大小為何?"}]})

In [ ]:
print(result["messages"][-1].content)

### 使用Streamlit建立基於Agent的聊天機器人

- Agent 將部屬於 Langserve 上
- Agent 在 Langserve上的應用有些眉角需要注意: 要額外使用pydantic數據模型註明input 和 output 類型

首先確認如何透過requests和langserve交流

將message寫成一個python dictionary, 包含兩個keys: `role`和`content`, 這樣 API 會收到一個`messages`，它是由 dict 組成的列表（裡面有 type + content 欄位），LangServe 就能將這個物件送入agent裡。

In [ ]:
import json
import requests
from textwrap import dedent

langserve_url = "http://localhost:9001/chatbot/invoke"

In [ ]:
chat_history = [{"role": "user", "content": "前天天氣如何?"}, 
                {"role": "ai", "content": "前天颳風下雨"}, 
                {"role": "user", "content": "我剛剛的問題為何?"}]

response = requests.post(
    langserve_url,
    json={'input': {"messages": chat_history}}
)

In [ ]:
response

In [ ]:
json.loads(response.text)

In [ ]:
response = requests.post(
    langserve_url,
    json={'input': {"messages": [{"role": "user", "content": "給予一個邊長為10的正方形，選定兩個對角點，在正方形內各畫一個半徑為10的1/4圓。請問兩個1/4圓重疊的部分大小為何?"}]}}
)

In [ ]:
json.loads(response.text)

## 內容編輯（Context Editing）

在對話變得很長、並且包含許多工具呼叫（tool calls）時，系統可能會超出模型的「可處理字元上限」。  
內容編輯功能能在這種情況下自動清除較舊的工具執行結果，只保留近期的重要內容，讓對話維持在可管理的範圍內。

內容編輯功能適用於以下情況：

- **對話很長、包含大量工具呼叫，導致超出 Token 限制**
- **降低 Token 成本**：移除已不再需要的舊工具輸出內容
- **只保留最近 N 筆工具執行結果**

---

## 設定參數（Configuration Parameters）

### `trigger`
- **類型：** number  
- **預設值：** `100000`  
- **說明：**  
  當對話累計 Token 數超過這個值時，系統會開始清除舊的工具輸出。

---

### `clear_at_least`
- **類型：** number  
- **預設值：** `0`  
- **說明：**  
  每次觸發清理時，至少要釋放的 Token 數量。  
  若設定為 `0`，則清除到足夠為止。

---

### `keep`
- **類型：** number  
- **預設值：** `3`  
- **說明：**  
  必須保留的最近工具結果數量。  
  這些工具結果永遠不會被清除。

---

### `clear_tool_inputs`
- **類型：** boolean  
- **預設值：** `False`  
- **說明：**  
  是否清除工具呼叫的輸入參數。  
  若設定為 `True`，則工具呼叫參數會被替換為空物件。

---

### `exclude_tools`
- **類型：** list[string]  
- **預設值：** `()`  
- **說明：**  
  不會被清除的工具清單。  
  列出的工具其輸出會永遠保留。

---

### `placeholder`
- **類型：** string  
- **預設值：** `[cleared]`  
- **說明：**  
  清除工具輸出後，用來取代原內容的佔位文字。

In [ ]:
from langchain.agents.middleware import ContextEditingMiddleware, ClearToolUsesEdit


agent = create_agent(
    model=model,
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=2000,
                    keep=3,
                    clear_tool_inputs=False,
                    exclude_tools=[],
                    placeholder="[cleared]",
                ),
            ],
        ),
    ],
)

## Advanced: 自訂義 Middleware

https://langchain-5e9cc07a.mintlify.app/oss/python/langchain/middleware/custom#node-style-hooks

### Node 風格的 Hooks

在特定的執行節點依序執行。適用於記錄（logging）、驗證（validation）與狀態更新（state updates）。可用的 hooks：

- **before_agent** - 在 agent 開始前執行（每次呼叫僅一次）
- **before_model** - 在每次模型呼叫前執行
- **after_model** - 在每次模型回應後執行
- **after_agent** - 在 agent 完成後執行（每次呼叫僅一次）

---

### Wrap 風格的 Hooks

攔截執行流程並控制何時呼叫處理器。適用於重試（retries）、快取（caching）與轉換（transformation）。  
你可以決定處理器被呼叫的次數：

- **0 次**（短路，short-circuit）
- **1 次**（正常流程）
- **多次**（重試邏輯）

可用的 hooks：

- **wrap_model_call** - 包住每一次模型呼叫
- **wrap_tool_call** - 包住每一次工具呼叫

---

### 建立 Middleware 的兩種方式

你可以用以下兩種方式建立 middleware：

#### 1. 基於裝飾器（Decorator-based）的 Middleware

快速且簡單，適合單一 hook 的 middleware。使用裝飾器來包裝個別函式。

#### 2. 基於類別（Class-based）的 Middleware

更強大，適合需要多個 hooks 或設定的複雜 middleware。

---

### 基於裝飾器的 Middleware

快速且簡單，適合單一 hook 的 middleware。使用裝飾器來包裝個別函式。可用的裝飾器：

#### Node 風格：

- **@before_agent** - 在 agent 開始前執行（每次呼叫僅一次）
- **@before_model** - 在每次模型呼叫前執行
- **@after_model** - 在每次模型回應後執行
- **@after_agent** - 在 agent 完成後執行（每次呼叫僅一次）

#### Wrap 風格：

- **@wrap_model_call** - 以自訂邏輯包住每一次模型呼叫
- **@wrap_tool_call** - 以自訂邏輯包住每一次工具呼叫


In [ ]:
result

In [ ]:
result

加入 ImMemoryStore。在dynamic_prompt中也可以加入ImMemoryStore

https://docs.langchain.com/oss/python/langchain/context-engineering#store-2

說實話這不算是一個好的範例，會在有調用工具的情況下出問題

In [ ]:
from textwrap import dedent
from dataclasses import dataclass
from typing import Callable, List

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore


# ---------------------------
# Context schema
# ---------------------------

@dataclass
class Context:
    user_id: str


# ---------------------------
# Middleware
# ---------------------------

@wrap_model_call
def inject_writing_style(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Inject user's email writing style from Store."""
    user_id = request.runtime.context.user_id

    # Read from Store: get user's writing style examples
    store = request.runtime.store
    writing_style = store.get(("writing_style",), user_id)

    print("inject_writing_style\n****")
    # for message in request.messages:
    #     print(f"type: {message.type}: {message.content}")
    # print("****")
    
    if writing_style:
        style = writing_style.value
        style_context = dedent(f"""\
        Your writing style:
        - Tone: {style.get('tone', 'professional')}
        - Typical greeting: "{style.get('greeting', 'Hi')}"
        - Typical sign-off: "{style.get('sign_off', 'Best')}"
        - Example email you've written:
        {style.get('example_email', '')}
        """)
        # print("*****")
        # print("🔥 inject_writing_style called")
        # print(f"messages count: {len(request.messages)}")
        # print(f"style_context: {style_context}")
        # print("*****")
        
        messages = [
            *request.messages,
            # {"role": "user", "content": style_context}
        ]
        request = request.override(messages=messages)

    return handler(request)


# ---------------------------
# Example tool
# ---------------------------

class EmailInput(BaseModel):
    to: str = Field(description="The recipient of the email")
    subject: str = Field(description="The email subject")
    body: str = Field(description="The email content")


@tool
def draft_email(input_: EmailInput) -> str:
    """Draft an email given recipient, subject, and body."""
    return f"To: {input_.to}\nSubject: {input_.subject}\n\n{input_.body}"


# ---------------------------
# Store + agent setup
# ---------------------------

store = InMemoryStore()

agent = create_agent(
    model="gpt-4o-mini",
    tools=[draft_email],
    middleware=[inject_writing_style],
    context_schema=Context,
    store=store,
)


# ---------------------------
# Seed the store with a user's style
# ---------------------------

user_id = "user_123"

store.put(
    ("writing_style",),
    user_id,
    {
        "tone": "friendly but professional",
        "greeting": "Hey there",
        "sign_off": "Cheers",
        "example_email": (
            "Hey there John,\n\n"
            "Just wanted to follow up on our chat yesterday. "
            "Let me know if you need anything else.\n\n"
            "Cheers,\nAlex"
        ),
    },
)
# ---------------------------
# Invoke the agent
# ---------------------------

# response = agent.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "Write a short email to Sarah about the project update."
#             }
#         ]
#     },
#     context=Context(user_id=user_id),
# )

# print(response)

跑了幾次後發現，有時候Agent笨笨的，不知道哪個時候要跳出迴圈

所以我們強制Agent在達成目標後離開

In [ ]:
from typing import Literal

from langgraph.runtime import Runtime
from langchain.agents.middleware import after_model, AgentState
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser


class IfGoal(BaseModel):
    name: Literal["Yes", "No"] = Field(description="If we already achieve the goal. The answer is either `Yes` or `No`")
    reason: str = Field(description="The reason behind the decision")


output_parser = PydanticOutputParser(pydantic_object=IfGoal)
format_instructions = output_parser.get_format_instructions()


model_check = ChatOpenAI(openai_api_key=os.environ['OPENAI_API_KEY'],
                        model_name="gpt-4o-mini", temperature=0,
                        name="check_goal_model")


pipeline_check = model_check|output_parser


@after_model(can_jump_to=["end"])
def check_goal(state: AgentState, runtime: Runtime):

    # # 方便起見
    # prompt = dedent(f"""
    # Last message: {state["messages"][-1].content}

    # Goal: Write a short email to Sarah about the project update.
    # output instructions: {format_instructions}
    # """)

    # result = pipeline_check.invoke(prompt)
    
    # if result.name == "Yes":
    #     return {"messages": state['messages'],
    #             "jump_to": "end"
    #             }
    # else:
    #     print("^^^")
    #     for message in state["messages"]:
    #         print(f"type: {message.type}: {message.content}")
    #     print("^^^")
    #     return None
    print("^^^")
    for message in state["messages"]:
        print(f"type: {message.type}: {message.content}")
    print("^^^")
    
    return None

In [ ]:
agent = create_agent(
    model="gpt-4o-mini",
    tools=[draft_email],
    middleware=[inject_writing_style, 
                check_goal
               ],
    context_schema=Context,
    store=store,
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Write a short email to Sarah about the project update."
            }
        ]
    },
    context=Context(user_id=user_id),
)

In [ ]:
response["messages"]

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I access a database?"}]
})

### 在代理防護措施（agent guardrails）之後

你也可以在代理完成防護檢查之後加入自訂中介軟體。  
這個階段的中介軟體通常用來處理已通過安全驗證的內容，例如：

- 調整或重新格式化代理產生的回應  
- 添加額外的資訊（如摘要、標籤、使用者提示）  
- 對回應進行記錄、分析或追蹤  
- 在必要時進行二次檢查或轉換（例如翻譯、簡化內容）  

放置在這個階段的中介軟體最適合做「後處理」，因為訊息已經通過安全與驗證流程，不會跨越防護規則。








In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based guardrail: Use an LLM to evaluate response safety."""

    def __init__(self):
        super().__init__()
        self.safety_model = init_chat_model("gpt-4o-mini")

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the final AI response
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a model to evaluate safety
        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])
        
        if "UNSAFE" in result.content:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "I cannot provide that response. Please rephrase your request."
                }],
                "jump_to": "end"
            }

        return None


agent = create_agent(
    model=model,
    middleware=[SafetyGuardrailMiddleware()],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I make explosives?"}]
})

In [ ]:
response = agent.invoke({"messages": [human_message]})

In [ ]:
response.results[0].categories.model_dump()

In [ ]:
response.results[0].category_scores.model_dump()

# 執行時環境（Runtime）

## 概覽

LangChain 的 `create_agent` 在底層是運行於 LangGraph 的執行時環境（runtime）上。  

LangGraph 提供了一個 **Runtime** 物件，其中包含以下資訊：

- **Context（上下文）**：靜態資訊，例如使用者 ID、資料庫連線，或其他代理程式執行時需要的依賴項目  
- **Store（存儲）**：一個 `BaseStore` 實例，用於長期記憶  
- **Stream writer（串流寫入器）**：用於透過「custom」串流模式傳輸資訊的物件  

### 執行時上下文的用途

Runtime 上下文可以為你的 `tools` 和 `middleware` 提供依賴注入（dependency injection）。  
- 不需要硬編碼值或依賴全域變數  
- 可以在呼叫代理程式時注入必要的依賴，例如資料庫連線、使用者 ID 或設定值  
- 讓工具更容易測試、重複使用且具彈性  

你可以在 **tools** 和 **middleware** 中存取這些執行時資訊，以動態調整行為或處理邏輯。


### Inside tools

#### Context

Access immutable configuration and contextual data like user IDs, session details, or application-specific configuration through runtime.context.

In [ ]:
from dataclasses import dataclass
from pydantic import BaseModel, Field
from langchain.tools import tool, ToolRuntime


USER_DATABASE = {
    "lsp1": {
        "preference": "an alluring and seductive tone.",
    },
    "lsp2": {
        "preference": "suggestive content",
    }
}


@dataclass
class Context:
    user_id: str


@tool
def fetch_user_email_preferences(runtime: ToolRuntime[Context]) -> str:  
    """Fetch the user's email preferences from the store."""
    user_id = runtime.context.user_id  

    preference = USER_DATABASE[user_id]
    
    return preference


agent = create_agent(
    model="gpt-4o-mini",
    context_schema=Context,
    tools=[fetch_user_email_preferences]
)


agent.invoke(
    {"messages": [{"role": "user", "content": "write an email to the user based on his preference."}]},
    context=Context(user_id="lsp1")  
)

#### 記憶體（Memory / Store）

你可以透過 **store** 在多次對話間存取持久化資料。  

- 使用方式：透過 `runtime.store` 存取  
- 功能：可以儲存與取得使用者專屬或應用程式專屬的資料  
- 優點：讓代理程式能記住過去的互動、偏好設定，或任何需要跨對話保留的資訊

In [ ]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"


# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."


# 關鍵是 InMemoryStore
store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})

#### 串流寫入器（Stream Writer）

你可以使用 `runtime.stream_writer` 在工具執行時即時傳送自訂更新。  

- 功能：讓工具的執行狀態可以即時回饋給使用者  
- 用途：提供使用者即時進度、動作更新或中間結果  
- 優點：改善使用者體驗，讓他們清楚了解工具正在做什麼

In [ ]:
@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    # Stream custom updates as the tool executes
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")

    return f"It's always sunny in {city}!"


agent = create_agent(
    model,
    tools=[get_weather],
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "How is the weather in Taipei"}]
})

A dummy example of combining Tool BaseTool class and the ToolRuntime.

We want to check if we can access the context with BaseTool class.

In [ ]:
import os

os.chdir("../../")

In [ ]:
from src.initialization import credential_init

credential_init()

In [ ]:
from pydantic import BaseModel, Field
from textwrap import dedent
from typing import Literal

from langchain.agents import create_agent
from langchain.tools import BaseTool, ToolRuntime
from langchain_core.output_parsers import PydanticOutputParser


class Inputs(BaseModel):
    city: str = Field(description="name of the city")


class DummyContext(BaseModel):
    weather: str


class DummyTool(BaseTool):

    input_output_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=Inputs)
    input_format_instruction: str = input_output_parser.get_format_instructions()
    
    name:str = "weather-reporter"
    description_template:str = dedent("""
    This tool can be used to report the weather of a city.
    The inputs contains the `city`.
    input format instructions: {input_format_instruction}
    """)

    description: str = description_template.format(input_format_instruction=input_format_instruction)
    
    def _run(self, 
             runtime: ToolRuntime[DummyContext], 
             **input):
        
        city = input["city"]

        weather = runtime.context.weather

        return f"{city}: {weather}"
    
    def _arun(self, runtime: ToolRuntime[DummyContext], **input):
        raise NotImplementedError("This tool does not support async")


agent = create_agent(
    model="gpt-4o-mini",
    context_schema=DummyContext,
    tools=[DummyTool()]
)


response = agent.invoke(
    {"messages": [{"role": "user", "content": "how is the weather of Taipei?"}]},
    context=DummyContext(weather="cloudy")  
)